# M2 Session 5 — Code organization, environments & GitHub collaboration

**Course:** Introduction to Digital Content and AI · Universitat de València  
**Date:** 2026-05-25  
**Duration:** 2.5 h practice (after 2 h theory)

---

## How to use this notebook

This notebook is a **guide and reference**, not a walkthrough to run cell by cell. Most of your work today will happen in VS Code, your terminal, and GitHub — not here.

Use it to:
- Check the expected project structure
- Copy-paste commands when needed
- Verify your outputs at each phase

Work in **pairs or threes**. Only **one person per team** creates the GitHub repository; everyone else is a collaborator.

---

## Team setup

Before you start, decide:

- Who creates the repository? → **Person A (repo owner)**
- Who works on data loading? → **Person A**
- Who works on visualization? → **Person B**
- Who builds the project website? → **Person C** (or split between A and B if working in pairs)

Write your team assignments here:

| Role | Name | GitHub username |
|---|---|---|
| Repo owner (Person A) | | |
| Visualization (Person B) | | |
| Website (Person C) | | |

---

## Phase 1 — Project scaffold

### Step 1 · Person A: create the GitHub repository

1. Go to [github.com](https://github.com) → click **+** → **New repository**
2. Name it `hotel-insights`
3. Set it to **Public**
4. Check **Add a README file** and select **Python** in the `.gitignore` template
5. Click **Create repository**

Then: *Settings → Collaborators → Add people* → invite your teammates by GitHub username.

<!-- 📸 (see slide for screenshot of GitHub "Create repository" page) -->

---

### Step 2 · Everyone: clone the repository

Wait for the invitation email, accept it, then clone via SSH:

```bash
git clone git@github.com:OWNER-USERNAME/hotel-insights.git
cd hotel-insights
```

Replace `OWNER-USERNAME` with Person A's GitHub username.

> **First time using SSH?** Follow the SSH setup steps on the Git & GitHub slides before continuing.

---

### Step 3 · Person A: create the project structure

Inside the `hotel-insights` folder, create the following directories and files:

```
hotel-insights/
├── data/
│   └── raw/
├── notebooks/
├── src/
│   └── hotel_insights/
│       └── __init__.py        ← create this (can be empty)
├── results/
│   └── figures/
└── docs/
    └── index.html             ← placeholder for Phase 4
```

You can create them all at once in the terminal:

```bash
mkdir -p data/raw data/processed notebooks src/hotel_insights results/figures docs
touch src/hotel_insights/__init__.py
```

---

### Step 4 · Person A: create `pyproject.toml`

Create `pyproject.toml` in the project root:

```toml
[project]
name = "hotel-insights"
version = "0.1.0"
description = "Exploratory analysis of hotel booking demand"
authors = [
    { name = "Your Name", email = "your.email@example.com" },
]
requires-python = ">=3.10"
dependencies = [
    "pandas>=2.0",
    "matplotlib>=3.7",
    "seaborn>=0.13",
    "jupyter>=1.0",
]

[project.optional-dependencies]
dev = ["pytest", "ruff"]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
```

> Update `authors` with all team members!

---

### Step 5 · Everyone: create and activate a virtual environment

Each team member does this **independently** on their own machine:

```bash

# Inside the hotel-insights folder:
python -m venv .venv

# Activate (Windows):
.venv\Scripts\activate

# Activate (macOS/Linux):
source .venv/bin/activate

# Install the project and all dependencies:
pip install -e .
```

In VS Code: open the command palette (`Ctrl+Shift+P`) → *Python: Select Interpreter* → choose `.venv`.

Verify it works:

```python
import pandas as pd
import matplotlib.pyplot as plt
print(f"pandas {pd.__version__} — ready ✅")
```

---

### Step 6 · Person A: update `README.md` and commit the scaffold

Replace the auto-generated `README.md` with a proper one (see the Python boilerplate slides for the template).

Then commit and push everything:

```bash
git add .
git commit -m "chore: initial project scaffold"
git push
```

**Everyone else: pull** the scaffold now:

```bash
git pull
```

At this point everyone has the same project structure. ✅

---

## Phase 2 — Feature branches

Each person now works on their own branch. **Do not push to `main` directly.**

---

### Person A — data loading module

```bash
git switch -c feature/data-loading
```

**Your task:** create `src/hotel_insights/data_loading.py` with a function that loads and lightly cleans the hotel bookings dataset.

Download the dataset:  
👉 [Hotel Booking Demand on Kaggle](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)  
Place the CSV at `data/raw/hotel_bookings.csv`.

```python

# src/hotel_insights/data_loading.py

import pandas as pd

DATA_PATH = "data/raw/hotel_bookings.csv"

def load_bookings(path=DATA_PATH):
    """Load the Hotel Booking Demand CSV and do basic cleaning.

    Returns a DataFrame with:
    - arrival_date column parsed as datetime
    - total_nights column (stays_in_weekend_nights + stays_in_week_nights)
    - rows with is_canceled == 1 and 0 both included
    """
    df = pd.read_csv(path)

# Parse arrival date (year + month only — day defaults to 1st)
    df["arrival_date"] = pd.to_datetime(
        df["arrival_date_year"].astype(str)
        + "-" + df["arrival_date_month"]
        + "-01",
        format="%Y-%B-%d"
    )

# Derived column
    df["total_nights"] = (
        df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
    )

    return df
```

Then create `notebooks/01_data_exploration.ipynb` (as a `.md` file, converted with `md2ipynb`) with:
- A cell importing `load_bookings`
- A call to `df.head()`, `df.info()`, `df.describe()`
- A brief markdown cell commenting on the shape and key columns

When done, commit and push:

```bash
git add .
git commit -m "feat: add load_bookings() and initial exploration notebook"
git push -u origin feature/data-loading
```

---

### Person B — visualization module

```bash
git switch -c feature/visualization
```

**Your task:** create `src/hotel_insights/visualization.py` with at least two plotting functions.

```python

# src/hotel_insights/visualization.py

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

PALETTE = "Blues_d"


def plot_monthly_bookings(df: pd.DataFrame, save_path=None):
    """Bar chart of total bookings by arrival month."""
    monthly = (
        df.groupby("arrival_date_month")
        .size()
        .reset_index(name="bookings")
    )

# Reorder months chronologically
    month_order = [
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"
    ]
    monthly["arrival_date_month"] = pd.Categorical(
        monthly["arrival_date_month"], categories=month_order, ordered=True
    )
    monthly = monthly.sort_values("arrival_date_month")

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(
        data=monthly, x="arrival_date_month", y="bookings",
        palette=PALETTE, ax=ax
    )
    ax.set_title("Total bookings by month")
    ax.set_xlabel("Month")
    ax.set_ylabel("Number of bookings")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig


def plot_cancellation_rate(df: pd.DataFrame, save_path=None):
    """Bar chart of cancellation rate by hotel type."""
    rates = (
        df.groupby("hotel")["is_canceled"]
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={"is_canceled": "cancellation_rate_%"})
    )

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.barplot(
        data=rates, x="hotel", y="cancellation_rate_%",
        palette=PALETTE, ax=ax
    )
    ax.set_title("Cancellation rate by hotel type")
    ax.set_xlabel("")
    ax.set_ylabel("Cancellation rate (%)")
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig
```

Then create `notebooks/02_visualizations.ipynb` that:
- Imports `load_bookings` and both chart functions
- Calls each function and displays the chart
- Saves figures to `results/figures/`

When done:

```bash
git add .
git commit -m "feat: add visualization module with two chart functions"
git push -u origin feature/visualization
```

---

### Person C (or A+B together) — project website

```bash
git switch -c feature/website
```

**Your task:** create `docs/index.html` — a simple, clean project page.

```html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Hotel Insights</title>
  <style>
    body {
      font-family: 'Segoe UI', sans-serif;
      max-width: 800px;
      margin: 40px auto;
      padding: 0 20px;
      color: #1e293b;
      line-height: 1.6;
    }
    h1 { color: #065a82; }
    h2 { color: #0b3c5d; border-bottom: 2px solid #d6e6f2; padding-bottom: 4px; }
    .team { display: flex; gap: 20px; flex-wrap: wrap; }
    .card {
      background: #f0f7ff;
      border-radius: 8px;
      padding: 16px;
      min-width: 180px;
    }
    a { color: #065a82; }
  </style>
</head>
<body>
  <h1>🏨 Hotel Insights</h1>
  <p>
    Exploratory analysis of hotel booking demand data, built for the
    <a href="https://www.uv.es">Universitat de València</a>
    AI in Tourism microcredential (2026).
  </p>

  <h2>About</h2>
  <p>
    We analyse the
    <a href="https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand">
      Hotel Booking Demand dataset</a>
    to uncover seasonal patterns, cancellation drivers, and guest segments.
  </p>

  <h2>Notebooks</h2>
  <ul>
    <li><a href="https://github.com/OWNER/hotel-insights/blob/main/notebooks/01_data_exploration.ipynb">Data exploration</a></li>
    <li><a href="https://github.com/OWNER/hotel-insights/blob/main/notebooks/02_visualizations.ipynb">Visualizations</a></li>
  </ul>

  <h2>Team</h2>
  <div class="team">
    <div class="card">
      <strong>Alice</strong><br>
      <a href="mailto:alice@example.com">alice@example.com</a>
    </div>
    <div class="card">
      <strong>Bob</strong><br>
      <a href="mailto:bob@example.com">bob@example.com</a>
    </div>
    <div class="card">
      <strong>Carol</strong><br>
      <a href="mailto:carol@example.com">carol@example.com</a>
    </div>
  </div>

  <h2>Repository</h2>
  <p>
    <a href="https://github.com/OWNER/hotel-insights">
      github.com/OWNER/hotel-insights
    </a>
  </p>
</body>
</html>
```

> Update the team names, emails, and GitHub URLs with your real information!

When done:

```bash
git add docs/
git commit -m "docs: add project website index.html"
git push -u origin feature/website
```

---

## Phase 3 — Pull Requests & code review

### Opening a PR

1. Go to your repository on GitHub
2. GitHub shows a yellow banner: *"feature/your-branch had recent pushes"* → click **Compare & pull request**
3. Write a clear title (e.g. `feat: add data loading module`) and a short description
4. Assign a teammate as reviewer
5. Click **Create pull request**

### Reviewing a PR

1. Open the PR assigned to you
2. Click **Files changed**
3. Read through the code — leave at least **one comment** (question, suggestion, or compliment)
4. Use the **Suggest a change** button to propose a specific fix inline
5. Submit your review as **Approve** or **Request changes**

### Merging

Once approved, the author clicks **Squash and merge** → **Confirm**.

Then everyone updates their local `main`:

```bash
git switch main
git pull
```

---

## Phase 4 — Release & GitHub Pages

### Create a release

```bash

# Make sure you're on main and up to date
git switch main
git pull

# Create an annotated tag
git tag -a v1.0.0 -m "First release — data loading, visualization, and project website"
git push origin v1.0.0
```

On GitHub: **Releases → Draft a new release** → select `v1.0.0` → write release notes → **Publish release**.

> Ask Copilot to help: *"Write release notes for v1.0.0 based on these commit messages: ..."*

---

### Enable GitHub Pages

1. On GitHub: **Settings → Pages**
2. Source: **Deploy from a branch**
3. Branch: `main` · Folder: `/docs`
4. Click **Save**
5. Wait ~30 seconds → a green banner shows your live URL

Your site is now live at:  
`https://OWNER-USERNAME.github.io/hotel-insights/`

---

## Deliverable

At the end of the session, share in the course chat:

1. **GitHub repository URL** — e.g. `https://github.com/alice/hotel-insights`
2. **GitHub Pages URL** — e.g. `https://alice.github.io/hotel-insights/`
3. **Release page** — click Releases on the repo to verify v1.0.0 is there

---

## Reflection questions

Answer these in your own words (no code needed):

1. What is the difference between `git add`, `git commit`, and `git push`?
2. Why is it better to work on a feature branch instead of pushing directly to `main`?
3. What does `pip install -e .` do, and why is the `-e` flag useful during development?
4. What is the purpose of `pyproject.toml`? How does it differ from `requirements.txt`?
5. What happens inside GitHub after you push a new branch and then open a Pull Request?
6. When would you use a GitHub Actions workflow in a data science project?

---

## Bonus challenges (if you finish early)

- Add a **GitHub Actions CI workflow** (`.github/workflows/ci.yml`) that installs your project and runs `ruff check src/` on every push. See the slide for a template.
- Add a **third chart function** (`plot_adr_by_hotel_type`) to `visualization.py` via a new PR.
- Add a chart **screenshot** to `docs/index.html` (export a figure as PNG to `docs/`).
- Add a `tests/` folder with a single `test_data_loading.py` that checks `load_bookings()` returns a DataFrame with the expected columns.